In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
!pip install seqeval

In [8]:
from datasets import load_dataset
import pandas as pd 
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
import evaluate
import numpy as np
from seqeval.metrics import classification_report

In [9]:
!wget -c https://huggingface.co/datasets/cfilt/HiNER-original/resolve/main/data/train.json -O train.json

--2026-10-07 04:03:49--  https://huggingface.co/datasets/cfilt/HiNER-original/resolve/main/data/train.json
Resolving huggingface.co (huggingface.co)... 18.239.50.16, 18.239.50.103, 18.239.50.49, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.16|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6266a847fcf21ddcb0266c6e/c5ca2ef90055d74f63a6b7b6305b2cdd295528bec35a47694b579792c76a4c31?X-Xet-Cas-Uid=public&user_id=public&response-content-type=application%2Fjson%3B+charset%3Dutf-8&xip=AMyQ7ypn-zU&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27train.json%3B+filename%3D%22train.json%22%3B&Expires=1791349429&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjI2NmE4NDdmY2YyMWRkY2IwMjY2YzZlL2M1Y2EyZWY5MDA1NWQ3NGY2M2E2YjdiNjMwNWIyY2RkMjk1NTI4YmVjMzVhNDc2OTRiNTc5NzkyYzc2YTRjMzFcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LXR5cGU9K

In [29]:
df_train = pd.read_json('train.json')
display(df_train)

,id,tokens,ner_tags
0,0,"[इस, क़ानून, का, कई, संगठनों, ने, विरोध, किया,...","[22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 2..."
1,1,"[देर, रात, तक, जुहू, चौपाटी, में, यह, नजारा, आ...","[22, 22, 22, 4, 15, 22, 22, 22, 22, 22, 22]"
2,2,"[रामनगर, इगलास, ,, अलीगढ़, ,, उत्तर, प्रदेश, स...","[4, 4, 22, 4, 22, 4, 15, 22, 22, 22, 22]"
3,3,"[पहाड़िया, आदिवासी, विद्रोह, १७७२-८०, के, मध्य...","[22, 22, 22, 22, 22, 22, 22, 4, 22, 22, 22, 22..."
4,4,"[दूसरे, मुख्य, टाँके, महल, परिसर, में, स्थित, ...","[6, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22..."
...,...,...,...
75822,75822,"[मछुआरों, की, नाव, .]","[22, 22, 22, 22]"
75823,75823,"[चरणदासियों, का, मंदिर, भी, बहुत, पुराना, है, ...","[4, 22, 22, 22, 22, 22, 22, 22, 22, 8, 22, 8, ..."
75824,75824,"[भैडियाणा,, गैरसैण, तहसील, में, भारत, के, उत्त...","[4, 4, 22, 22, 4, 22, 4, 22, 22, 22, 4, 22, 22..."
75825,75825,"[ज़िन्दगी, (, अंग्रेज़ी:, Life), 1976, में, बन...","[22, 22, 2, 22, 10, 22, 22, 2, 13, 22, 22, 22]"


In [30]:
df_train = df_train[:10000]

In [31]:
names = [
        "B-FESTIVAL",
        "B-GAME",
        "B-LANGUAGE",
        "B-LITERATURE",
        "B-LOCATION",
        "B-MISC",
        "B-NUMEX",
        "B-ORGANIZATION",
        "B-PERSON",
        "B-RELIGION",
        "B-TIMEX",
        "I-FESTIVAL",
        "I-GAME",
        "I-LANGUAGE",
        "I-LITERATURE",
        "I-LOCATION",
        "I-MISC",
        "I-NUMEX",
        "I-ORGANIZATION",
        "I-PERSON",
        "I-RELIGION",
        "I-TIMEX",
        "O"
]

In [32]:
index_to_name = {i: name for i, name in enumerate(names)}
name_to_index = {name: i for i, name in enumerate(names)}

In [33]:
df_exploded = df_train[0:10].explode(['tokens', 'ner_tags'])
df_exploded["ner_tags"] = df_exploded["ner_tags"].map(index_to_name)

In [34]:
display(df_exploded)

,id,tokens,ner_tags
0,0,इस,O
0,0,क़ानून,O
0,0,का,O
0,0,कई,O
0,0,संगठनों,O
...,...,...,...
9,9,साल,I-TIMEX
9,9,की,O
9,9,सज़ा,O
9,9,सुनाई,O


In [35]:
train_df, temp_df = train_test_split(df_train, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [36]:
ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False)
})

In [37]:
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
label_all_tokens = False

In [38]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=256,
    )
    new_labels = []
    for i, word_ids in enumerate(tokenized.word_ids(batch_index=j) for j in range(len(examples["tokens"]))):
        labels = []
        word_labels = examples["ner_tags"][i]
        prev_word_id = None
        for w_id in word_ids:
            if w_id is None:
                labels.append(-100)
            elif w_id != prev_word_id:
                labels.append(int(word_labels[w_id]))
            else:
                labels.append(int(word_labels[w_id]) if label_all_tokens else -100)
            prev_word_id = w_id
        new_labels.append(labels)
    tokenized["labels"] = new_labels
    return tokenized

In [39]:
encoded_ds = ds.map(tokenize_and_align_labels,
                    batched=True,
                    remove_columns=ds["train"].column_names
                )

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [40]:
model = AutoModelForTokenClassification.from_pretrained(model_name, 
                                                        num_labels=len(index_to_name),
                                                        id2label=index_to_name,
                                                        label2id=name_to_index)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ig

In [41]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [42]:
training_args = TrainingArguments(
    output_dir="ner-hindi",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to=[],
)


In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_ds["train"],
    eval_dataset=encoded_ds["validation"],
    data_collator=data_collator,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.283664,2.180284
2,1.513728,1.453235


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
predictions, labels, _ = trainer.predict(encoded_ds["test"])
predictions = np.argmax(predictions, axis=-1)
 
true_preds, true_labels = [], []
for pred, lab in zip(predictions, labels):
    cur_preds, cur_labels = [], []
    for p, l in zip(pred, lab):
        if l != -100:
            cur_preds.append(index_to_name[p])
            cur_labels.append(index_to_name[l])
    true_preds.append(cur_preds)
    true_labels.append(cur_labels)
 
# use your original names list (tags like "O", "B-PERSON", etc.)
print(classification_report(true_labels, true_preds, digits=4))

              precision    recall  f1-score   support

        GAME     0.0000    0.0000    0.0000         5
    LANGUAGE     0.0000    0.0000    0.0000        34
  LITERATURE     0.0000    0.0000    0.0000         4
    LOCATION     0.7253    0.8661    0.7894       881
        MISC     0.0000    0.0000    0.0000        33
       NUMEX     0.0000    0.0000    0.0000        90
ORGANIZATION     0.0000    0.0000    0.0000       119
      PERSON     0.1581    0.2575    0.1959       167
    RELIGION     0.0000    0.0000    0.0000         8
       TIMEX     0.0000    0.0000    0.0000        90

   micro avg     0.6088    0.5632    0.5851      1431
   macro avg     0.0883    0.1124    0.0985      1431
weighted avg     0.4650    0.5632    0.5089      1431



/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
